# Generate, then judge: ask your own question about a database

One 7B model (Qwen2.5-Coder-7B-Instruct) with two small adapters answers questions about the BIRD dev databases in SQL. For each question it writes 17 queries, runs them, has its verifier read every query **and what it returned**, and answers with the result the verifier trusts most. The page shows the SQL, the result, and every query it considered.

**How to run it**
1. *Runtime > Change runtime type*: **T4 GPU** (free) or **L4 / A100**. On an L4 or A100 the system is exactly the one in the report; on a T4 the base model is loaded in 4 bit to fit, so answers can differ slightly.
2. *Runtime > Run all*. The first run takes about 10 minutes: it installs the code, downloads the model (15 GB), the two adapters and BIRD dev (350 MB, from BIRD's own site, checked against the files used in every experiment).
3. The page appears at the bottom of this notebook, and as a public link ending in `gradio.live` that opens it in a full browser tab. An answer takes a few seconds on an A100, longer on a T4.

In [ ]:
REPO_URL = "https://github.com/selimsidan/generate-then-judge-sql"
REF = "main"            # branch, tag or commit to run
ADAPTERS = "hf"         # "hf": the public Hugging Face repository; "drive": the author's Drive (before publishing)
DRIVE_RUNS = "/content/drive/MyDrive/586/effsql/runs"

In [ ]:
import os, subprocess, sys

REPO_DIR = "/content/generate-then-judge-sql"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REF], check=True)
if subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}"],
                  capture_output=True).returncode == 0:
    subprocess.run(["git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}"], check=True)
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + "[gen,demo]"],
                     capture_output=True, text=True)
if pip.returncode:
    raise RuntimeError("pip install failed:\n" + pip.stdout[-3000:] + pip.stderr[-3000:])
print("commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
if ADAPTERS == "drive":
    from google.colab import drive
    drive.mount("/content/drive")

### Get everything ready (downloads, databases, model)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
from effsql.app import launch, prepare

app = prepare(adapters=ADAPTERS, drive_runs=DRIVE_RUNS if ADAPTERS == "drive" else None)
print("Running on", app.mode)

### The page

In [ ]:
launch(app, share=True)

When you are done: *Runtime > Disconnect and delete runtime*, so the GPU stops using compute units.